In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
path = Path(r"C:\Users\HP\Desktop\First-Django-backend\predicthub_backend\ml\data\synthetic")

df = pd.read_csv(path / "trades.csv")

df.head()

,id,user_id,market_id,outcome_type,trade_type,amount_staked,tokens_amount,price_at_execution,onchain_trade_id,onchain_tx_hash,created_at
0,9518,343,43,YES,buy,5.69,9.22,0.617188,704744,0x08a6ee02814c,2025-12-01T00:44:53.699582+00:00
1,9517,343,24,NO,buy,27.48,72.52,0.378937,682598,0x0bbbaa05fa91,2025-12-01T00:14:51.699582+00:00
2,9516,343,21,NO,buy,19.25,35.01,0.549838,884372,0x02b7d408b64c,2025-11-30T23:37:36.699582+00:00
3,11367,443,41,YES,buy,18.41,33.20,0.554495,609407,0x0d269f02c62a,2025-11-30T23:35:45.772966+00:00
4,11366,443,18,NO,buy,13.51,44.56,0.303211,861836,0x03063a0d26f2,2025-11-30T23:18:43.772966+00:00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9572 entries, 0 to 9571
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  9572 non-null   int64  
 1   user_id             9572 non-null   int64  
 2   market_id           9572 non-null   int64  
 3   outcome_type        9572 non-null   object 
 4   trade_type          9572 non-null   object 
 5   amount_staked       9572 non-null   float64
 6   tokens_amount       9572 non-null   float64
 7   price_at_execution  9572 non-null   float64
 8   onchain_trade_id    9572 non-null   int64  
 9   onchain_tx_hash     9572 non-null   object 
 10  created_at          9572 non-null   object 
dtypes: float64(3), int64(4), object(4)
memory usage: 822.7+ KB


In [5]:
df.describe()

,id,user_id,market_id,amount_staked,tokens_amount,price_at_execution,onchain_trade_id
count,9572.000000,9572.000000,9572.000000,9572.000000,9572.000000,9572.000000,9572.000000
mean,8790.500000,308.501567,29.433243,72.417925,153.405080,0.500590,505414.335562
std,2763.342722,144.000026,8.623605,113.923291,251.919615,0.116451,290465.760630
min,4005.000000,57.000000,15.000000,1.000000,1.530000,0.300035,1271.000000
25%,6397.750000,186.000000,22.000000,16.030000,32.175000,0.398251,255357.000000
50%,8790.500000,306.000000,29.000000,30.795000,61.080000,0.500001,505504.000000
75%,11183.250000,433.000000,37.000000,45.352500,103.355000,0.601226,760024.500000
max,13576.000000,556.000000,44.000000,499.840000,1661.650000,0.699996,999763.000000


In [6]:
df['created_at'] = pd.to_datetime(df['created_at'])

In [7]:
df = df.sort_values(['user_id', 'created_at']).reset_index(drop=True)

In [8]:
df['time_since_last_trade'] = df.groupby('user_id')['created_at'].diff().dt.total_seconds()

In [9]:
df['time_since_last_trade'] = df['time_since_last_trade'].fillna(0)

In [10]:
df['time_since_last_trade'] = df['time_since_last_trade'].astype(int)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9572 entries, 0 to 9571
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   id                     9572 non-null   int64              
 1   user_id                9572 non-null   int64              
 2   market_id              9572 non-null   int64              
 3   outcome_type           9572 non-null   object             
 4   trade_type             9572 non-null   object             
 5   amount_staked          9572 non-null   float64            
 6   tokens_amount          9572 non-null   float64            
 7   price_at_execution     9572 non-null   float64            
 8   onchain_trade_id       9572 non-null   int64              
 9   onchain_tx_hash        9572 non-null   object             
 10  created_at             9572 non-null   datetime64[ns, UTC]
 11  time_since_last_trade  9572 non-null   int64            

In [12]:
df

,id,user_id,market_id,outcome_type,trade_type,amount_staked,tokens_amount,price_at_execution,onchain_trade_id,onchain_tx_hash,created_at,time_since_last_trade
0,4005,57,26,NO,buy,11.92,21.13,0.564013,293136,0x07e6bd06c72f,2025-11-23 09:03:51.031124+00:00,0
1,4006,57,17,NO,sell,32.43,70.52,0.459871,348235,0x0f06420a1af6,2025-11-23 10:03:01.031124+00:00,3550
2,4007,57,18,NO,buy,29.45,42.47,0.693420,626550,0x0223530342a4,2025-11-23 10:05:04.031124+00:00,123
3,4008,57,26,NO,sell,30.70,60.84,0.504569,200312,0x07b0620ac054,2025-11-23 10:34:53.031124+00:00,1789
4,4009,57,16,NO,buy,26.48,42.53,0.622604,986937,0x0c83fd0d0969,2025-11-23 10:52:26.031124+00:00,1053
...,...,...,...,...,...,...,...,...,...,...,...,...
9567,13572,556,24,YES,sell,15.68,29.02,0.540299,259864,0x0e73be02860d,2025-11-27 17:19:05.731050+00:00,2079
9568,13573,556,42,YES,buy,19.99,29.77,0.671548,622360,0x0e06f60abb33,2025-11-27 18:18:53.731050+00:00,3588
9569,13574,556,35,YES,sell,21.01,35.31,0.594968,839050,0x07a1ea0b9281,2025-11-27 18:23:41.731050+00:00,288
9570,13575,556,27,NO,sell,4.16,12.06,0.344885,327160,0x0969c20efdb0,2025-11-27 18:25:43.731050+00:00,122
